## **HOME CREDIT DEFAULT RISK** 
### INGESTÃO DE CSV PARA ORACLE

Le os CSVs originais do dataset, faz limpeza minima de tipos/colunas
e carrega em lote (bulk insert) nas tabelas ja criadas pelos script da pasta de SQL.

### 1. Imports e Instalações

In [1]:
!pip install pandas oracledb 
import logging
from pathlib import Path

import numpy as np
import oracledb
import pandas as pd

### 2. Configurações

In [13]:
DB_USER = "HOME_CREDIT"
DB_PASSWORD = "senha123"
DB_DSN = "localhost:1521/XEPDB1"

DATA_PATH = Path(r"C:\Users\lucas\Documents\Python\Case Default Risk\home-credit-default-risk")

conn = oracledb.connect(
    user=DB_USER,
    password=DB_PASSWORD,
    dsn=DB_DSN
)

### 3. Funções de Carga

In [48]:

def insert_dataframe(df, table_name, connection, chunksize=5000):
    df = df.copy()

    # Troca inf/-inf por nulo
    df = df.replace([np.inf, -np.inf], np.nan)

    # Converte tudo para object para permitir None real
    df = df.astype(object)

    # Troca NaN/pd.NA por None
    df = df.where(pd.notnull(df), None)

    cols = list(df.columns)
    placeholders = ", ".join([f":{i+1}" for i in range(len(cols))])
    col_names = ", ".join(cols)

    sql = f"""
        INSERT INTO {table_name} ({col_names})
        VALUES ({placeholders})
    """

    cursor = connection.cursor()

    try:
        for start in range(0, len(df), chunksize):
            batch = df.iloc[start:start + chunksize]
            cursor.executemany(sql, batch.values.tolist())
            connection.commit()
            print(f"{table_name}: {start + len(batch)} linhas carregadas")

    except Exception as e:
        connection.rollback()
        print(f"Erro ao carregar tabela {table_name}")
        print(f"Lote iniciado na linha: {start}")
        raise e

    finally:
        cursor.close()


# application_train + application_test
train = pd.read_csv(DATA_PATH / "application_train.csv")
test = pd.read_csv(DATA_PATH / "application_test.csv")

train["DATASET_TYPE"] = "TRAIN"
test["DATASET_TYPE"] = "TEST"
test["TARGET"] = None

application = pd.concat([train, test], ignore_index=True)

insert_dataframe(application[app_cols], "HC_APPLICATION", conn)

# bureau
bureau = pd.read_csv(DATA_PATH / "bureau.csv")
bureau_cols = [
    "SK_ID_BUREAU", "SK_ID_CURR", "CREDIT_ACTIVE", "CREDIT_CURRENCY",
    "DAYS_CREDIT", "CREDIT_DAY_OVERDUE", "DAYS_CREDIT_ENDDATE",
    "DAYS_ENDDATE_FACT", "AMT_CREDIT_MAX_OVERDUE", "CNT_CREDIT_PROLONG",
    "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_LIMIT",
    "AMT_CREDIT_SUM_OVERDUE", "CREDIT_TYPE", "DAYS_CREDIT_UPDATE"
]
insert_dataframe(bureau[bureau_cols], "HC_BUREAU", conn)


# bureau_balance
def insert_csv_in_chunks(file_path, table_name, columns, connection, chunksize=100000):
    cursor = connection.cursor()

    col_names = ", ".join(columns)
    placeholders = ", ".join([f":{i+1}" for i in range(len(columns))])

    sql = f"""
        INSERT INTO {table_name} ({col_names})
        VALUES ({placeholders})
    """

    total = 0

    for chunk in pd.read_csv(file_path, usecols=columns, chunksize=chunksize):
        chunk = chunk.replace([np.inf, -np.inf], np.nan)
        chunk = chunk.astype(object)
        chunk = chunk.where(pd.notnull(chunk), None)

        cursor.executemany(sql, chunk.values.tolist())
        connection.commit()

        total += len(chunk)
        print(f"{table_name}: {total} linhas carregadas")

    cursor.close()

pos_cols = [
    "SK_ID_PREV", "SK_ID_CURR", "MONTHS_BALANCE", "CNT_INSTALMENT",
    "CNT_INSTALMENT_FUTURE", "NAME_CONTRACT_STATUS", "SK_DPD", "SK_DPD_DEF"
]

inst_cols = [
    "SK_ID_PREV", "SK_ID_CURR", "NUM_INSTALMENT_VERSION",
    "NUM_INSTALMENT_NUMBER", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT",
    "AMT_INSTALMENT", "AMT_PAYMENT"
]

cc_cols = [
    "SK_ID_PREV", "SK_ID_CURR", "MONTHS_BALANCE", "AMT_BALANCE",
    "AMT_CREDIT_LIMIT_ACTUAL", "AMT_DRAWINGS_CURRENT",
    "AMT_PAYMENT_CURRENT", "AMT_TOTAL_RECEIVABLE",
    "CNT_DRAWINGS_CURRENT", "NAME_CONTRACT_STATUS", "SK_DPD", "SK_DPD_DEF"
]


bureau_balance = pd.read_csv(DATA_PATH / "bureau_balance.csv")
insert_csv_in_chunks(
    DATA_PATH / "bureau_balance.csv",
    "HC_BUREAU_BALANCE",
    ["SK_ID_BUREAU", "MONTHS_BALANCE", "STATUS"],
    conn,
    chunksize=50000
)


# previous_application
previous_cols = [
    "SK_ID_PREV", "SK_ID_CURR", "NAME_CONTRACT_TYPE", "AMT_ANNUITY",
    "AMT_APPLICATION", "AMT_CREDIT", "AMT_DOWN_PAYMENT",
    "NAME_CONTRACT_STATUS", "DAYS_DECISION", "CODE_REJECT_REASON",
    "NAME_CLIENT_TYPE", "NAME_PRODUCT_TYPE", "CHANNEL_TYPE"
]

# insert_csv_in_chunks(
    DATA_PATH / "previous_application.csv",
    "HC_PREVIOUS_APPLICATION",
    previous_cols,
    conn,
    chunksize=50000
)

# POS_CASH_balance
insert_csv_in_chunks(
    DATA_PATH / "POS_CASH_balance.csv",
    "HC_POS_CASH_BALANCE",
    pos_cols,
    conn,
    chunksize=100000
)

# installments_payments
insert_csv_in_chunks(
    DATA_PATH / "installments_payments.csv",
    "HC_INSTALLMENTS_PAYMENTS",
    inst_cols,
    conn,
    chunksize=30000
)

# credit_card_balance
insert_csv_in_chunks(
    DATA_PATH / "credit_card_balance.csv",
    "HC_CREDIT_CARD_BALANCE",
    cc_cols,
    conn,
    chunksize=30000
)

conn.close()

HC_CREDIT_CARD_BALANCE: 30000 linhas carregadas
HC_CREDIT_CARD_BALANCE: 60000 linhas carregadas
HC_CREDIT_CARD_BALANCE: 90000 linhas carregadas
HC_CREDIT_CARD_BALANCE: 120000 linhas carregadas
HC_CREDIT_CARD_BALANCE: 150000 linhas carregadas
HC_CREDIT_CARD_BALANCE: 180000 linhas carregadas
HC_CREDIT_CARD_BALANCE: 210000 linhas carregadas
HC_CREDIT_CARD_BALANCE: 240000 linhas carregadas
HC_CREDIT_CARD_BALANCE: 270000 linhas carregadas
HC_CREDIT_CARD_BALANCE: 300000 linhas carregadas
HC_CREDIT_CARD_BALANCE: 330000 linhas carregadas
HC_CREDIT_CARD_BALANCE: 360000 linhas carregadas
HC_CREDIT_CARD_BALANCE: 390000 linhas carregadas
HC_CREDIT_CARD_BALANCE: 420000 linhas carregadas
HC_CREDIT_CARD_BALANCE: 450000 linhas carregadas
HC_CREDIT_CARD_BALANCE: 480000 linhas carregadas
HC_CREDIT_CARD_BALANCE: 510000 linhas carregadas
HC_CREDIT_CARD_BALANCE: 540000 linhas carregadas
HC_CREDIT_CARD_BALANCE: 570000 linhas carregadas
HC_CREDIT_CARD_BALANCE: 600000 linhas carregadas
HC_CREDIT_CARD_BALANCE:

### 4. Execução do Rollback em casos de erro na execução

No Python

In [49]:
# conn.rollback()

In [50]:
# try:
#     cursor.close()
# except:
#     pass

# conn.close()

In [51]:
# conn = oracledb.connect(
#     user=DB_USER,
#     password=DB_PASSWORD,
#     dsn=DB_DSN
#  )

No SQL

In [ ]:
# TRUNCATE TABLE HC_CREDIT_CARD_BALANCE;

# ALTER TABLE HC_CREDIT_CARD_BALANCE
# DROP CONSTRAINT FK_HC_CC_PREV;